# 🎯 Parameter Optimization / 参数优化

Welcome back! Now that you've run your first backtest, it's time to find the "Goldilocks" settings for your strategy. 🐻

In this notebook, we'll explore:
1.  **Grid Search**: Checking every combination in a parameter grid. 🔍
2.  **Walk-Forward Analysis**: Simulating how your strategy would have been re-optimized over time. 🚶

我们不仅要运行策略，还要找到最适合它的参数。在本示例中，我们将演示：
1.  **网格搜索**：地毯式搜索最优参数组合。🔍
2.  **Walk-Forward 分析**：模拟实战中的参数动态调整过程。🚶


In [ ]:
import numpy as np
import pandas as pd

from quanteval import DualMAStrategy, GridSearch, WalkForwardAnalysis

In [ ]:
index = pd.date_range('2021-01-04', periods=260, freq='B')
close = pd.Series(80 + np.linspace(0, 30, len(index)) + np.sin(np.arange(len(index)) / 8) * 3, index=index)
data = pd.DataFrame({
    'Open': close * 0.99,
    'High': close * 1.01,
    'Low': close * 0.98,
    'Close': close,
    'Volume': np.linspace(500, 1200, len(index)),
    'Amount': close * 800,
}, index=index)
data['Ret'] = data['Close'].pct_change().fillna(0.0)

### 🔍 1. Grid Search / 网格搜索

Let's test all combinations of fast and slow periods. ⚡

我们要遍历所有的快慢周期组合，看看哪个最赚钱。⚡


In [ ]:
param_grid = {
    'fast_window': [5, 10],
    'slow_window': [20, 30],
    'ma_type': ['sma'],
}
search = GridSearch(DualMAStrategy, param_grid, verbose=False)
search.fit(data).all_results.head()

### 🚶 2. Walk-Forward Analysis / 样本外循环分析

Wait! Optimization alone is dangerous—ever heard of **overfitting**? 😱 
Walk-Forward simulates picking the "best" parameters from earlier data and seeing how they perform on a tiny bit of "future" data.

警告：单纯的优化可能导致“过拟合”。😱 
Walk-Forward 模拟了在实战中如何基于历史数据更新参数，并观察在小部分“未来数据”上的真实表现。


In [ ]:
analysis = WalkForwardAnalysis(DualMAStrategy, param_grid, train_period=120, test_period=40, verbose=False)
wf_result = analysis.run(data)
print(wf_result.summary())